# Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('../src'))
from D_Compute_Spectrograms.load_interpolation_config import load_interpolation_config
from E_Plot_Spectrograms.load_average_tfr_results import load_average_tfr_results
from E_Plot_Spectrograms.plot_average_tfr import plot_average_tfr
from E_Plot_Spectrograms.load_matlab_colormap import load_matlab_colormap
from E_Plot_Spectrograms.average_grouped_tfr import average_grouped_tfr
from E_Plot_Spectrograms.create_grouped_interpolation_config import create_grouped_interpolation_config

# Variables

In [ ]:
subjectID = 'Pilote001'
date      = '2026-07-09'
task      = 'Task1_PPS'
data_path =  '/Users/floremunier/Library/CloudStorage/Dropbox-NeuroRestore/Flore Munier-Jolain/PercepPD/Data'

baseline_time_start = -0.5
N_baseline_points = 500

### Load Data

In [ ]:
file_path_raw = os.path.join(data_path, subjectID, task, "Output", "TFR_raw", f"{subjectID}_{date}_{task}_raw_tfr_average")
tfr_average_dict_raw = load_average_tfr_results(file_path_raw)

config_path_raw = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_per_trial_type.json')
interpolation_config_raw = load_interpolation_config(config_path_raw)

file_path_merged = os.path.join(data_path, subjectID, task, "Output", "TFR_merged", f"{subjectID}_{date}_{task}_merged_tfr_single")
tfr_single_dict_merged = load_average_tfr_results(file_path_merged)

config_path_merged = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'D_Compute_Spectrograms', 'interpolation_config_for_merging.json')
interpolation_config_merged = load_interpolation_config(config_path_merged)

colormap_path = os.path.join(os.path.dirname(data_path), 'Code', 'EEG_analysis', 'Scripts', 'src', 'E_Plot_Spectrograms', 'NRcolormap.mat')
NRcmap = load_matlab_colormap(colormap_path, "NRcolormap")

### Plot Non-merged Spectrograms

In [ ]:
output_path_raw = os.path.join(data_path, subjectID, task, 'Output', 'Spectrograms_raw')

for event_name in tfr_average_dict_raw.keys():
    plot_average_tfr(
        tfr_average_dict=tfr_average_dict_raw,
        event_name=event_name,
        interpolation_config=interpolation_config_raw,
        n_baseline_pts=N_baseline_points,
        output_path=output_path_raw,
        colormin=-5,
        colormax=5,
        cmap=NRcmap
    )

### Plot merged Spectrograms

In [ ]:
groups = {
    "Both_Near_Fast": ["Both_D1_Narrow_Fast", "Both_D2_Narrow_Fast", "Both_D3_Narrow_Fast"],
    "Both_Near_Slow": ["Both_D1_Narrow_Slow", "Both_D2_Narrow_Slow", "Both_D3_Narrow_Slow"],

    "Both_Far_Fast": ["Both_D6_Narrow_Fast", "Both_D5_Narrow_Fast", "Both_D4_Narrow_Fast"],
    "Both_Far_Slow": ["Both_D6_Narrow_Slow", "Both_D5_Narrow_Slow", "Both_D4_Narrow_Slow"],

    "VibrotactileOnly_Near_Fast": ["VibrotactileOnly_D1_Narrow_Fast", "VibrotactileOnly_D2_Narrow_Fast", "VibrotactileOnly_D3_Narrow_Fast"],
    "VibrotactileOnly_Near_Slow": ["VibrotactileOnly_D1_Narrow_Slow", "VibrotactileOnly_D2_Narrow_Slow", "VibrotactileOnly_D3_Narrow_Slow"],

    "VibrotactileOnly_Far_Fast": ["VibrotactileOnly_D6_Narrow_Fast", "VibrotactileOnly_D5_Narrow_Fast", "VibrotactileOnly_D4_Narrow_Fast"],
    "VibrotactileOnly_Far_Slow": ["VibrotactileOnly_D6_Narrow_Slow", "VibrotactileOnly_D5_Narrow_Slow", "VibrotactileOnly_D4_Narrow_Slow"],
}

# Average tfr per defined groups
tfr_average_merged = average_grouped_tfr(tfr_single_dict_merged,groups)

# Create a new timimg dict of the events timimg for the grouped data (for plot use)
grouped_interpolation_config = create_grouped_interpolation_config(groups,interpolation_config_merged)

In [ ]:
output_path_merged = os.path.join(data_path, subjectID, task, 'Output', 'Spectrograms_merged')
# Plot the averaged tfr
for event_name in tfr_average_merged.keys():
    plot_average_tfr(
        tfr_average_dict=tfr_average_merged,
        event_name=event_name,
        interpolation_config=grouped_interpolation_config,
        n_baseline_pts=N_baseline_points,
        output_path=output_path_merged,
        colormin=-6,
        colormax=1.5,
        cmap=NRcmap,
        groups=groups
    )